# Silver Transformation — TfL Arrivals

Transform raw TfL arrivals snapshots into structured arrival-prediction observations.

Each row represents one arrival prediction observed during one API request.

Repeated observations of the same train across different snapshots are preserved because they provide useful operational history.

This notebook:

1. Reads Bronze arrivals.
2. Parses JSON with an explicit schema.
3. Flattens arrival predictions.
4. Normalizes timestamps.
5. Applies data-quality rules.
6. Creates a deterministic observation key.
7. Removes duplicate rows within a snapshot.
8. Merges validated observations into Silver.

**Source:** `workspace.urbanpulse_bronze.tfl_arrivals`

**Target:** `workspace.urbanpulse_silver.tfl_arrivals`

## 1. Initialise project paths

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

In [0]:
from pyspark.sql import functions as F

from urbanpulse.transformations.tfl_arrivals import (
    transform_tfl_arrivals,
)

from urbanpulse.quality.tfl_arrivals import (
    valid_tfl_arrivals,
    invalid_tfl_arrivals,
)

from urbanpulse.utils.delta import (
    merge_insert_only,
)

## 3. Define source and target tables

In [0]:
BRONZE_TABLE = (
    "workspace."
    "urbanpulse_bronze."
    "tfl_arrivals"
)

SILVER_TABLE = (
    "workspace."
    "urbanpulse_silver."
    "tfl_arrivals"
)

## 4. Read Bronze arrivals

Each Bronze row represents one station API request and contains the complete arrivals array returned by TfL.

In [0]:
bronze_df = spark.table(
    BRONZE_TABLE
)

bronze_count = bronze_df.count()

print(
    f"Bronze station requests: "
    f"{bronze_count}"
)

display(
    bronze_df.select(
        "request_id",
        "source_endpoint",
        "ingested_at",
        "http_status",
    )
    .orderBy(
        F.col("ingested_at").desc()
    )
)

## 5. Parse and flatten arrival predictions

The Bronze payload is a JSON array.

Each array element becomes one structured Silver candidate row.

In [0]:
parsed_df = transform_tfl_arrivals(
    bronze_df
)

parsed_count = parsed_df.count()

print(
    f"Parsed arrival observations: "
    f"{parsed_count}"
)

display(
    parsed_df
    .orderBy(
        F.col("snapshot_at").desc()
    )
)

## 6. Inspect parsed arrival fields

Review identifiers, timestamps, platform information, destination, and predicted time to station before applying quality rules.

In [0]:
display(
    parsed_df.select(
        "request_id",
        "requested_stop_point_id",
        "arrival_id",
        "vehicle_id",
        "station_name",
        "line_name",
        "platform_name",
        "destination_name",
        "time_to_station_seconds",
        "prediction_timestamp",
        "expected_arrival",
        "snapshot_at",
    )
    .orderBy(
        "station_name",
        "expected_arrival",
    )
)

## 7. Validate timestamp parsing

TfL timestamp strings are converted to Spark timestamps before entering Silver.

In [0]:
timestamp_check_df = (
    parsed_df
    .select(
        F.count("*").alias(
            "total_rows"
        ),

        F.sum(
            F.col(
                "prediction_timestamp"
            ).isNull().cast("int")
        ).alias(
            "missing_prediction_timestamp"
        ),

        F.sum(
            F.col(
                "expected_arrival"
            ).isNull().cast("int")
        ).alias(
            "missing_expected_arrival"
        ),
    )
)

display(timestamp_check_df)

## 8. Apply data-quality rules

Valid arrival observations require:

- request ID
- requested station ID
- arrival ID
- line ID
- station name
- prediction timestamp
- expected arrival timestamp
- non-negative `time_to_station_seconds`
- snapshot timestamp

In [0]:
valid_df = valid_tfl_arrivals(
    parsed_df
)

invalid_df = invalid_tfl_arrivals(
    parsed_df
)

valid_count = valid_df.count()
invalid_count = invalid_df.count()

print(
    f"Valid observations: "
    f"{valid_count}"
)

print(
    f"Invalid observations: "
    f"{invalid_count}"
)

## 9. Inspect invalid arrival observations

Operational APIs can contain incomplete records.

Invalid rows are inspected before deciding whether they indicate a source-contract failure or an acceptable edge case.

In [0]:
if invalid_count > 0:
    display(
        invalid_df.select(
            "request_id",
            "requested_stop_point_id",
            "arrival_id",
            "vehicle_id",
            "station_name",
            "line_id",
            "platform_name",
            "prediction_timestamp",
            "expected_arrival",
            "time_to_station_seconds",
        )
    )
else:
    print(
        "No invalid arrival observations."
    )

## 10. Create a deterministic observation key

The same TfL arrival prediction can appear in multiple polling snapshots.

UrbanPulse therefore identifies an observation using both the API request and the TfL arrival ID.

A SHA-256 key provides a compact deterministic identifier for downstream joins.

In [0]:
keyed_df = (
    valid_df
    .withColumn(
        "arrival_observation_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("request_id"),
                F.col("arrival_id"),
            ),
            256,
        ),
    )
)

In [0]:
display(
    keyed_df.select(
        "arrival_observation_key",
        "request_id",
        "arrival_id",
        "vehicle_id",
        "station_name",
    )
)

## 11. Deduplicate observations

Duplicate records within the same API response are removed using:

`request_id + arrival_id`

Records from different API requests remain separate historical observations.

In [0]:
deduplicated_df = (
    keyed_df
    .dropDuplicates([
        "request_id",
        "arrival_id",
    ])
)

print(
    f"Rows before deduplication: "
    f"{valid_count}"
)

print(
    f"Rows after deduplication: "
    f"{deduplicated_df.count()}"
)

## 12. Add Silver processing metadata

In [0]:
silver_df = (
    deduplicated_df
    .withColumn(
        "processed_at",
        F.current_timestamp(),
    )
)

## 13. Verify observation-key uniqueness

The generated observation key must be unique within the dataset being written.

In [0]:
duplicate_keys_df = (
    silver_df
    .groupBy(
        "arrival_observation_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_key_count = (
    duplicate_keys_df.count()
)

if duplicate_key_count > 0:
    display(duplicate_keys_df)

    raise ValueError(
        "Duplicate arrival observation "
        "keys detected."
    )

print(
    "Arrival observation keys are unique."
)

## 14. Merge observations into Silver

Delta `MERGE` prevents the same Bronze arrival observation from being inserted twice when this notebook is rerun.

In [0]:
merge_result = merge_insert_only(
    spark=spark,
    source_df=silver_df,
    target_table=SILVER_TABLE,
    merge_condition="""
        target.arrival_observation_key
        =
        source.arrival_observation_key
    """,
)

print(
    f"Silver arrivals table: "
    f"{merge_result}"
)

## 15. Verify Silver arrivals

Inspect the structured operational arrival observations.

In [0]:
%sql
SELECT
    arrival_observation_key,
    requested_stop_point_id,
    arrival_id,
    vehicle_id,
    station_name,
    line_name,
    platform_name,
    destination_name,
    time_to_station_seconds,
    prediction_timestamp,
    expected_arrival,
    snapshot_at,
    processed_at
FROM workspace.urbanpulse_silver.tfl_arrivals
ORDER BY snapshot_at DESC, station_name, expected_arrival;

## 16. Review arrival statistics

This query shows monitored stations that returned at least one arrival observation during the collected snapshots. Stations returning an empty TfL arrivals array will not appear in Silver.

In [0]:
%sql
SELECT
    COUNT(*) AS observations,
    COUNT(DISTINCT request_id) AS api_requests,
    COUNT(DISTINCT requested_stop_point_id) AS monitored_stations,
    COUNT(DISTINCT line_id) AS lines,
    COUNT(DISTINCT vehicle_id) AS vehicles
FROM workspace.urbanpulse_silver.tfl_arrivals;

In [0]:
%sql
SELECT
    requested_stop_point_id,
    MAX(station_name) AS station_name,
    COUNT(*) AS observations,
    COUNT(DISTINCT line_id) AS lines,
    COUNT(DISTINCT vehicle_id) AS vehicles,
    MIN(time_to_station_seconds) AS minimum_eta_seconds,
    MAX(time_to_station_seconds) AS maximum_eta_seconds
FROM workspace.urbanpulse_silver.tfl_arrivals
GROUP BY requested_stop_point_id
ORDER BY station_name;

In [0]:
%sql
SELECT
    line_id,
    line_name,
    COUNT(*) AS observations,
    COUNT(DISTINCT vehicle_id) AS vehicles
FROM workspace.urbanpulse_silver.tfl_arrivals
GROUP BY
    line_id,
    line_name
ORDER BY observations DESC;

In [0]:
%sql
SELECT *
FROM workspace.urbanpulse_silver.tfl_arrivals
WHERE time_to_station_seconds < 0;

In [0]:
%sql
SELECT
    arrival_observation_key,
    COUNT(*) AS records
FROM workspace.urbanpulse_silver.tfl_arrivals
GROUP BY arrival_observation_key
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT COUNT(*) AS observations
FROM workspace.urbanpulse_silver.tfl_arrivals;  

In [0]:
%sql
SELECT
    vehicle_id,
    station_name,
    line_name,
    platform_name,
    time_to_station_seconds,
    prediction_timestamp,
    expected_arrival,
    snapshot_at
FROM workspace.urbanpulse_silver.tfl_arrivals
WHERE vehicle_id IS NOT NULL
ORDER BY
    vehicle_id,
    snapshot_at;